In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import sparse
import matplotlib.pyplot as plt

In [163]:
df = pd.read_csv('subset.csv', index_col=0)

In [164]:
df = df.set_index(['Time', 'PIXEL_ID'])
df
#df = df.set_index(['Time'])

Longitude   Latitude  MaxTemperature  Precipitation  ET0  \
Time       PIXEL_ID                                                             
1952-01-01 1          5.419101  49.510015             NaN            9.6  NaN   
           2          5.488333  49.509469             NaN            8.1  NaN   
           3          5.557563  49.508882             NaN            5.1  NaN   
           4          5.626792  49.508253             NaN            4.8  NaN   
           5          5.419910  49.554969             NaN            7.4  NaN   
...                        ...        ...             ...            ...  ...   
1954-12-31 1356       4.517511  51.491625             2.8            0.0  NaN   
           1357       4.734134  51.491493             1.8            0.0  NaN   
           1358       4.806341  51.491362             1.8            0.0  NaN   
           1359       5.022959  51.490706             2.0            0.0  NaN   
           1360       5.095163  51.490399             2.0            0.0  NaN   

                     Year  
Time       PIXEL_ID        
1952-01-01 1         1952  
           2         1952  
           3         1952  
           4         1952  
           5         1952  
...                   ...  
1954-12-31 1356      1954  
           1357      1954  
           1358      1954  
           1359      1954  
           1360      1954  

[1490560 rows x 6 columns]

In [165]:
df_time_slice = df.loc['1952-01-01']

In [166]:
rows, current_row = [], []
previous_lon = -1000
for idx, row in df_time_slice.iterrows():
    lon_val = row.Longitude
    pixel_idx = row.name
    if lon_val > previous_lon:
        current_row.append(pixel_idx)
    else:
        rows.append(current_row)
        current_row = [pixel_idx]
    previous_lon = lon_val
rows.append(current_row)

In [167]:
for row in rows:
    df_time_slice.loc[df_time_slice.index.isin(row), 'Latitude'] = df_time_slice.loc[row].Latitude.mean()

In [168]:
sorted_lons = np.sort(np.unique(df_time_slice.Longitude))

cols, current_col = [], []
previous_lon = sorted_lons[0]
for lon in sorted_lons:
    if lon - previous_lon < 0.005:
        current_col.append(lon)
    else:
        cols.append(current_col)
        current_col = [lon]
    previous_lon = lon
cols.append(current_col)

In [169]:
for col in cols:
    df_time_slice.loc[df_time_slice.Longitude.isin(col), 'Longitude'] = df_time_slice.loc[df_time_slice.Longitude.isin(col), 'Longitude'].mean()

In [170]:
lons = np.unique(df_time_slice.Longitude)
lats = np.unique(df_time_slice.Latitude)

In [171]:
for pixel_idx in df_time_slice.index:
    df.loc[(slice(None), pixel_idx), 'Latitude'] = df_time_slice.loc[pixel_idx, 'Latitude']
    df.loc[(slice(None), pixel_idx), 'Longitude'] = df_time_slice.loc[pixel_idx, 'Longitude']

In [172]:
df = df.reset_index().drop(columns='PIXEL_ID')
df = df.set_index(['Time', 'Latitude', 'Longitude'])

In [173]:
ds = df.to_xarray()

In [174]:
ds

<xarray.Dataset>
Dimensions:         (Time: 1096, Latitude: 45, Longitude: 59)
Coordinates:
  * Time            (Time) object '1952-01-01' '1952-01-02' ... '1954-12-31'
  * Latitude        (Latitude) float64 49.51 49.55 49.6 ... 51.4 51.45 51.49
  * Longitude       (Longitude) float64 2.514 2.59 2.661 ... 6.282 6.353 6.42
Data variables:
    MaxTemperature  (Time, Latitude, Longitude) float64 nan nan nan ... nan nan
    Precipitation   (Time, Latitude, Longitude) float64 nan nan nan ... nan nan
    ET0             (Time, Latitude, Longitude) float64 nan nan nan ... nan nan
    Year            (Time, Latitude, Longitude) float64 nan nan nan ... nan nan

In [ ]:
ds.Precipitation[10].plot()